# RAG Architecture in Practice with OpenAI

### Why is RAG essential in the market?
1. **Always up-to-date data** - No need to retrain models
2. **Verifiable answers** - With sources of information
3. **Optimized cost** - Avoids expensive fine-tuning
4. **Real applications**: Support chatbots, document analysis, internal assistants

## Setup - Load API Key and Initialize Client

In [1]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print("API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

API key loaded successfully.


## 1. Traditional Prompting vs RAG - Practical Comparison

Let's see in practice the difference between using only an LLM (traditional prompting) and using RAG.

In [18]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model= "gpt-4o-mini",
    api_key= api_key,
    temperature=0,
)

In [19]:
# EXAMPLE 1: Traditional Prompting (without RAG)

question = "What is our company's work-from-home policy?"

traditional_prompt = ChatPromptTemplate.from_template(
    "Answer the following question: {question}"
)

In [21]:
traditional_chain = traditional_prompt | llm

traditional_response = traditional_chain.invoke({"question": question})

In [22]:
print(traditional_response.content)

I'm sorry, but I don't have access to specific company policies or internal documents. To find out your company's work-from-home policy, I recommend checking your employee handbook, company intranet, or reaching out to your HR department for the most accurate and up-to-date information.


-----
## NOTE: What is this | ?

The `|` operator in Python, in this context with **LangChain**, is an **elegant way to compose steps in a chain** of execution, similar to a **pipeline**.

### ✅ Practical meaning in LangChain:

```python
chain = prompt | llm
```

This code creates a **chain** where:

* The `prompt` is executed first,
* And the result (text formatted with the variables) is sent directly to the `llm` (language model, such as Gemini),
* Returning the **generated response**.

-------

## Loading Our Document
Now, instead of simulating the document, let's load the `politica_home_office.pdf` file located in the same folder.

In [31]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "../../data/politica_home_office.pdf"
loader = PyPDFLoader(pdf_path)

document = loader.load()

In [32]:
document

[Document(metadata={'producer': 'Skia/PDF m142 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'politica_home_office', 'source': '../../data/politica_home_office.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Política  de  Home  Office  –  Super  Dooper  Marketing   \n1.  Introdução  e  Objetivo  \nA  presente  Política  de  Home  Office  tem  como  objetivo  estabelecer  as  diretrizes  e  regras  para  \no\n \ntrabalho\n \nremoto\n \nna\n \nSuper\n \nDooper.\n \nNosso\n \nintuito\n \né\n \ngarantir\n \nque\n \na\n \nflexibilidade\n \ne\n \na\n \nautonomia\n \nproporcionadas\n \npor\n \neste\n \nmodelo\n \nandem\n \nde\n \nmãos\n \ndadas\n \ncom\n \na\n \nprodutividade,\n \na\n \nsegurança\n \ne\n \no\n \nbem-estar\n \nde\n \nnossos\n \ncolaboradores,\n \nmantendo\n \na\n \nexcelência\n \nque\n \nnos\n \ndefine\n \nno\n \nmercado\n \nde\n \nmarketing.\n \n2.  Modelo  de  Trabalho  \nO  modelo  de  trabalho  adotado  pela  Super  Dooper  é 

In [33]:
company_context = documento[0].page_content

In [34]:
company_context

'Política  de  Home  Office  –  Super  Dooper  Marketing   \n1.  Introdução  e  Objetivo  \nA  presente  Política  de  Home  Office  tem  como  objetivo  estabelecer  as  diretrizes  e  regras  para  \no\n \ntrabalho\n \nremoto\n \nna\n \nSuper\n \nDooper.\n \nNosso\n \nintuito\n \né\n \ngarantir\n \nque\n \na\n \nflexibilidade\n \ne\n \na\n \nautonomia\n \nproporcionadas\n \npor\n \neste\n \nmodelo\n \nandem\n \nde\n \nmãos\n \ndadas\n \ncom\n \na\n \nprodutividade,\n \na\n \nsegurança\n \ne\n \no\n \nbem-estar\n \nde\n \nnossos\n \ncolaboradores,\n \nmantendo\n \na\n \nexcelência\n \nque\n \nnos\n \ndefine\n \nno\n \nmercado\n \nde\n \nmarketing.\n \n2.  Modelo  de  Trabalho  \nO  modelo  de  trabalho  adotado  pela  Super  Dooper  é  o  híbrido .  Espera-se  que  os  \ncolaboradores\n \ncompareçam\n \nao\n \nescritório\n \npara\n \nreuniões\n \nde\n \nequipe,\n \nbrainstormings\n \ne\n \neventos\n \nde\n \nintegração,\n \nconforme\n \nagendamento\n \nprévio\n \ncom\n \no\n \ngestor\

In [35]:
print(company_context[:500] + "...")

Política  de  Home  Office  –  Super  Dooper  Marketing   
1.  Introdução  e  Objetivo  
A  presente  Política  de  Home  Office  tem  como  objetivo  estabelecer  as  diretrizes  e  regras  para  
o
 
trabalho
 
remoto
 
na
 
Super
 
Dooper.
 
Nosso
 
intuito
 
é
 
garantir
 
que
 
a
 
flexibilidade
 
e
 
a
 
autonomia
 
proporcionadas
 
por
 
este
 
modelo
 
andem
 
de
 
mãos
 
dadas
 
com
 
a
 
produtividade,
 
a
 
segurança
 
e
 
o
 
bem-estar
 
de
 
nossos
 
colaboradores,
 
mantendo
 
a
 
...


In [37]:
# Example 2: With RAG - Using the PDF Context

prompt_rag = ChatPromptTemplate.from_template("""
Use the context below to answer the question.
If you don't know the answer based on the context, say that the information is not available.

Context: {context}
Question: {question}

Answer:""")

In [38]:
chain_rag = prompt_rag | llm

rag_response = chain_rag.invoke({"context": company_context, "question": question})

In [39]:
print(rag_response.content)

The work-from-home policy at Super Dooper Marketing establishes guidelines for remote work, aiming to ensure that flexibility and autonomy align with productivity, security, and employee well-being. The company adopts a hybrid work model, where employees are expected to attend the office for team meetings, brainstorming sessions, and integration events as scheduled with their direct manager. The frequency of office attendance is determined by each team, balancing project needs with employee flexibility. The standard work hours are Monday to Friday, from 9 AM to 6 PM, with a one-hour lunch break, but flexible hours can be arranged with direct managers, provided communication and availability for meetings are maintained during standard business hours. Employees must record their work hours using an online system for transparency. Communication tools include Slack for instant messaging and Google Meet for video calls, while email is reserved for formal communications and client interactio